# Notebook 02 — Feature Extraction
### WID2003 Cognitive Science | FSKTM, Universiti Malaya

---

## Overview

Raw gaze coordinates change every millisecond and cannot be fed directly into a classifier. This notebook transforms the cleaned gaze data into a set of **meaningful, stable eye-tracking features** per participant per task — the feature matrix (`X`) that the models will learn from.

**Inputs**
| File | Description |
|---|---|
| `data/processed/metrics_clean.parquet` | Cleaned Metrics data (from Notebook 01) |
| `data/processed/raw_gaze_clean.parquet` | Cleaned raw gaze stream (from Notebook 01) |
| `data/external/task_correct_aoi_map.json` | Maps each task to its `answer` AOI and distractor AOIs |

**Output**
| File | Description |
|---|---|
| `data/processed/features_per_task.parquet` | One row per (participant × task), ~35 feature columns |

---

## Learning Objectives

By the end of this notebook, you should be able to:

1. Explain what an **eye-tracking feature** is and why raw gaze coordinates are not suitable as direct model inputs
2. Distinguish between **correct-AOI features** and **distractor features**, and explain the cognitive significance of each
3. Compute derived features including **scanpath length**, **re-fixation count**, and **AOI dwell ratio** from a raw gaze sequence
4. Explain how **pupil diameter change** can serve as a proxy for cognitive load over time

---

## Background

### What Is a Feature?

A **feature** is a measurable property that summarises the behaviour of one participant across a trial. Instead of thousands of gaze samples, we extract one number per participant per metric — a stable statistic that reflects how they searched.

### AOI-Based Feature Split

For each task, features are extracted separately for:
- The **`answer` AOI** — the correct target region
- The **distractor AOIs** (`M1`, `M2`, ...) — the wrong regions

This split lets us capture how much time a participant wasted on distractors compared to the target.

### Features Extracted

**From the Metrics file** (pre-computed by Tobii Pro Lab per AOI):

| Feature | Cognitive interpretation |
|---|---|
| `correct_Total_duration_of_fixations` | How long did the participant look at the target? |
| `correct_Number_of_fixations` | How many times did their gaze land on the target? |
| `correct_Time_to_first_fixation` | How quickly did they detect the target? |
| `correct_Number_of_Visits` | How many times did they return to the target? |
| `distractor_sum_Total_duration_of_fixations` | Total time spent on wrong regions |
| `Average_pupil_diameter` | Mean arousal/load during the trial |

**From the raw gaze stream** (computed in this notebook):

| Feature | How it is computed |
|---|---|
| `correct_aoi_dwell_ratio` | Frames with `AOI hit [task - answer] == 1` ÷ total valid frames |
| `distractor_dwell_ratio` | Frames on any M-AOI ÷ total valid frames |
| `scanpath_length` | Sum of pixel distances between consecutive fixation centroids |
| `refixation_count` | Number of `0 → 1` transitions in the `answer` AOI hit column |
| `pupil_dilation_change` | Mean pupil (2nd half of trial) − mean pupil (1st half) |

### The `correct_aoi_dwell_ratio`

This is the most important single feature in the study. It answers: *"Of all the time the participant looked at valid screen regions, what fraction was spent on the correct answer?"*

A **high-performing** student typically shows a high dwell ratio early — their gaze is drawn to the target quickly and stays there.

---

## Discussion Questions

1. Why do we use `correct_aoi_dwell_ratio` instead of `correct_Total_duration_of_fixations`? What does normalisation by total valid frames correct for?
2. A participant has a very low `correct_aoi_dwell_ratio` but still found the correct answer. What might explain this pattern?
3. What does a high `refixation_count` on the `answer` AOI suggest about the participant's cognitive process (e.g. uncertainty, verification strategy)?
4. Why might `pupil_dilation_change` be positive for harder tasks and negative or near-zero for easy tasks?
5. The `scanpath_length` is measured in pixels. What are the limitations of comparing scanpath lengths across participants who may have sat at different distances from the screen?

In [9]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
from scipy.spatial.distance import euclidean

from src.config import (
    METRICS_CLEAN_PKL, RAWGAZE_CLEAN_PKL, FEATURES_PER_TASK, AOI_MAP_JSON,
    MetricsCols, ExportCols, METRICS_FEATURE_COLS, TASKS
)

pd.set_option('display.max_columns', 50)

## 1. Load inputs

In [10]:
metrics = pd.read_parquet(METRICS_CLEAN_PKL)
gaze    = pd.read_parquet(RAWGAZE_CLEAN_PKL)

with open(AOI_MAP_JSON) as f:
    aoi_map = json.load(f)

# Strip _comment key
aoi_map = {k: v for k, v in aoi_map.items() if not k.startswith('_')}

print("AOI map loaded for tasks:", list(aoi_map.keys()))
print("Metrics shape:", metrics.shape)
print("Gaze shape:",    gaze.shape)

AOI map loaded for tasks: ['findDice', 'findYummy', 'frogInBathroom', 'headphoneInBathroom', 'frog', 'whoCheats', 'whoThief', 'spotNeedleInst']
Metrics shape: (1174, 57)
Gaze shape: (111212, 282)


## 2. Features from Metrics file

For each (Participant, Media/task), extract separate feature rows for:
- correct AOI
- all distractors (aggregated)
- total (all AOIs combined)

In [11]:
def extract_metrics_features(metrics_df, aoi_map):
    """Return a DataFrame with one row per (participant, task)."""
    records = []

    for task in TASKS:
        if task not in aoi_map:
            print(f"WARNING: {task} not in aoi_map, skipping")
            continue

        correct_aoi    = aoi_map[task]['correct_aoi']
        distractor_aois = aoi_map[task]['distractor_aois']

        task_df = metrics_df[metrics_df[MetricsCols.MEDIA].str.contains(task, na=False, case=False)]

        for participant, grp in task_df.groupby(MetricsCols.PARTICIPANT):
            row = {'participant_id': participant, 'task': task}

            # --- Correct AOI metrics ---
            correct_rows = grp[grp[MetricsCols.AOI].str.strip() == correct_aoi]
            if len(correct_rows) > 0:
                cr = correct_rows.iloc[0]
                for col in METRICS_FEATURE_COLS:
                    row[f'correct_{col}'] = cr.get(col, np.nan)
            else:
                for col in METRICS_FEATURE_COLS:
                    row[f'correct_{col}'] = np.nan

            # --- Distractor metrics (mean across distractor AOIs) ---
            dist_rows = grp[grp[MetricsCols.AOI].str.strip().isin(distractor_aois)]
            if len(dist_rows) > 0:
                for col in [MetricsCols.TOTAL_FIX_DUR, MetricsCols.NUM_FIXATIONS,
                            MetricsCols.TOTAL_VISIT_DUR, MetricsCols.NUM_VISITS]:
                    row[f'distractor_sum_{col}'] = dist_rows[col].sum()
                    row[f'distractor_mean_{col}'] = dist_rows[col].mean()
            else:
                for col in [MetricsCols.TOTAL_FIX_DUR, MetricsCols.NUM_FIXATIONS,
                            MetricsCols.TOTAL_VISIT_DUR, MetricsCols.NUM_VISITS]:
                    row[f'distractor_sum_{col}']  = np.nan
                    row[f'distractor_mean_{col}'] = np.nan

            # --- AOI transition count (proxy: total visits across all AOIs) ---
            row['total_aoi_visits'] = grp[MetricsCols.NUM_VISITS].sum()

            records.append(row)

    return pd.DataFrame(records)


metrics_features = extract_metrics_features(metrics, aoi_map)
print(f"Metrics features shape: {metrics_features.shape}")
metrics_features.head()

Metrics features shape: (56, 30)


,participant_id,task,correct_Total_duration_of_fixations,correct_Average_duration_of_fixations,correct_Number_of_fixations,correct_Time_to_first_fixation,correct_Duration_of_first_fixation,correct_Total_duration_of_Visit,correct_Number_of_Visits,correct_Time_to_first_Visit,correct_Average_duration_of_Visit,correct_Total_duration_of_Glances,correct_Number_of_Glances,correct_Number_of_saccades_in_AOI,correct_Peak_velocity_of_entry_saccade,correct_Peak_velocity_of_exit_saccade,correct_Average_pupil_diameter,correct_Average_eye_openness,correct_Time_to_first_mouse_click,correct_Time_from_first_fixation_to_mouse_click,correct_Number_of_mouse_clicks,distractor_sum_Total_duration_of_fixations,distractor_mean_Total_duration_of_fixations,distractor_sum_Number_of_fixations,distractor_mean_Number_of_fixations,distractor_sum_Total_duration_of_Visit,distractor_mean_Total_duration_of_Visit,distractor_sum_Number_of_Visits,distractor_mean_Number_of_Visits,total_aoi_visits
0,p001,findDice,1233,1233.0,1,4614.0,1233.0,1233,1,4614.0,1233.0,1250,1,0,98.85,196.34,4.92483,9.38149,NaN,NaN,0,26003,464.339286,71,1.267857,26352,470.571429,62,1.107143,117
1,p002,findDice,533,178.0,3,3868.0,217.0,533,3,3868.0,178.0,633,3,0,145.67,111.33,4.42462,8.92977,NaN,NaN,0,12250,218.750000,48,0.857143,12283,219.339286,47,0.839286,92
2,p003,findDice,1200,300.0,4,1249.0,433.0,1517,2,1249.0,758.0,1600,2,0,385.27,180.67,3.35311,8.63351,NaN,NaN,0,9193,164.160714,36,0.642857,9377,167.446429,26,0.464286,72
3,p004,findDice,1453,291.0,5,836.0,267.0,1503,2,836.0,752.0,1587,2,3,313.85,189.50,4.20383,9.59572,NaN,NaN,0,10390,185.535714,38,0.678571,10590,189.107143,35,0.625000,73
4,p005,findDice,1633,327.0,5,3659.0,83.0,1650,4,3659.0,413.0,1867,4,1,153.01,89.07,3.69877,5.76278,NaN,NaN,0,2900,51.785714,13,0.232143,2966,52.964286,12,0.214286,44


## 3. Features from raw gaze stream

Computed per (participant, task):
- `correct_aoi_dwell_ratio` — fraction of total valid frames on correct AOI
- `distractor_dwell_ratio` — fraction on any distractor AOI
- `scanpath_length` — cumulative Euclidean distance between consecutive fixation points
- `refixation_count` — number of returns to correct AOI after leaving
- `pupil_dilation_change` — pupil diameter second-half minus first-half

In [12]:
def build_aoi_hit_col(task, aoi):
    """Build the column name pattern used in the Data export: 'AOI hit [task - aoi]'."""
    return f"AOI hit [{task} - {aoi}]"


def compute_scanpath_length(fix_x, fix_y):
    """Cumulative Euclidean distance between consecutive fixation centroids."""
    coords = list(zip(fix_x, fix_y))
    if len(coords) < 2:
        return 0.0
    return sum(euclidean(coords[i], coords[i+1]) for i in range(len(coords)-1))


def count_refixations(aoi_hit_series):
    """Count how many times gaze returns to AOI after leaving (0->1 transitions)."""
    transitions = aoi_hit_series.diff().fillna(0)
    return int((transitions == 1).sum())


def extract_gaze_features(gaze_df, aoi_map):
    records = []

    for task in TASKS:
        if task not in aoi_map:
            continue

        correct_aoi     = aoi_map[task]['correct_aoi']
        distractor_aois = aoi_map[task]['distractor_aois']
        correct_col     = build_aoi_hit_col(task, correct_aoi)
        distractor_cols = [build_aoi_hit_col(task, a) for a in distractor_aois
                           if build_aoi_hit_col(task, a) in gaze_df.columns]

        if correct_col not in gaze_df.columns:
            print(f"WARNING: column '{correct_col}' not found — skipping {task}")
            continue

        task_df = gaze_df[gaze_df[ExportCols.PRESENTED_MEDIA].str.contains(task, na=False, case=False)]

        for participant, grp in task_df.groupby(ExportCols.PARTICIPANT_NAME):
            row = {'participant_id': participant, 'task': task}

            total_frames = len(grp)
            valid_frames = grp[ExportCols.GAZE_X].notna().sum()

            # Dwell ratios
            row['correct_aoi_dwell_ratio']    = grp[correct_col].sum() / max(valid_frames, 1)
            if distractor_cols:
                distractor_hits = grp[distractor_cols].any(axis=1).sum()
                row['distractor_dwell_ratio'] = distractor_hits / max(valid_frames, 1)
            else:
                row['distractor_dwell_ratio'] = np.nan

            # Scanpath length (fixation rows only)
            fix_rows = grp[grp[ExportCols.EYE_MOVEMENT_TYPE] == 'Fixation'].dropna(
                subset=[ExportCols.FIXATION_X, ExportCols.FIXATION_Y]
            )
            row['scanpath_length'] = compute_scanpath_length(
                fix_rows[ExportCols.FIXATION_X].values,
                fix_rows[ExportCols.FIXATION_Y].values
            )

            # Re-fixation count on correct AOI
            row['refixation_count'] = count_refixations(grp[correct_col])

            # Pupil dilation change (first half vs second half)
            pupil = grp[ExportCols.PUPIL_FILTERED].dropna()
            if len(pupil) >= 4:
                mid = len(pupil) // 2
                row['pupil_dilation_change'] = pupil.iloc[mid:].mean() - pupil.iloc[:mid].mean()
            else:
                row['pupil_dilation_change'] = np.nan

            row['valid_frames']  = int(valid_frames)
            row['total_frames']  = int(total_frames)

            records.append(row)

    return pd.DataFrame(records)


gaze_features = extract_gaze_features(gaze, aoi_map)
print(f"Gaze features shape: {gaze_features.shape}")
gaze_features.head()

Gaze features shape: (56, 9)


,participant_id,task,correct_aoi_dwell_ratio,distractor_dwell_ratio,scanpath_length,refixation_count,pupil_dilation_change,valid_frames,total_frames
0,p001,findDice,0.178388,0.221269,5790.550812,1,0.527158,583,820
1,p002,findDice,0.054908,0.109817,6927.737942,3,0.388697,601,745
2,p003,findDice,0.484211,0.000000,2381.134215,4,0.253926,190,236
3,p004,findDice,0.550633,0.000000,1832.249865,5,-0.108810,158,167
4,p005,findDice,0.250896,0.025090,5922.532484,5,0.082493,558,897


## 4. Merge Metrics and Gaze features

In [13]:
if gaze_features.empty:
    print("WARNING: gaze_features is empty — Data Export TSV AOI columns not found.")
    print("         Re-export the Data Export TSV from Tobii Pro Lab with renamed AOIs,")
    print("         then re-run this notebook. Proceeding with metrics-only features.")
    features = metrics_features.copy()
else:
    features = pd.merge(
        metrics_features,
        gaze_features,
        on=['participant_id', 'task'],
        how='outer'
    )

print(f"Combined features shape: {features.shape}")
print(f"Participants: {features['participant_id'].nunique()}")
print(f"Tasks: {features['task'].unique()}")
features.head()

Combined features shape: (56, 37)
Participants: 8
Tasks: ['findDice' 'findYummy' 'frog' 'frogInBathroom' 'headphoneInBathroom'
 'spotNeedleInst' 'whoCheats' 'whoThief']


,participant_id,task,correct_Total_duration_of_fixations,correct_Average_duration_of_fixations,correct_Number_of_fixations,correct_Time_to_first_fixation,correct_Duration_of_first_fixation,correct_Total_duration_of_Visit,correct_Number_of_Visits,correct_Time_to_first_Visit,correct_Average_duration_of_Visit,correct_Total_duration_of_Glances,correct_Number_of_Glances,correct_Number_of_saccades_in_AOI,correct_Peak_velocity_of_entry_saccade,correct_Peak_velocity_of_exit_saccade,correct_Average_pupil_diameter,correct_Average_eye_openness,correct_Time_to_first_mouse_click,correct_Time_from_first_fixation_to_mouse_click,correct_Number_of_mouse_clicks,distractor_sum_Total_duration_of_fixations,distractor_mean_Total_duration_of_fixations,distractor_sum_Number_of_fixations,distractor_mean_Number_of_fixations,distractor_sum_Total_duration_of_Visit,distractor_mean_Total_duration_of_Visit,distractor_sum_Number_of_Visits,distractor_mean_Number_of_Visits,total_aoi_visits,correct_aoi_dwell_ratio,distractor_dwell_ratio,scanpath_length,refixation_count,pupil_dilation_change,valid_frames,total_frames
0,p001,findDice,1233,1233.0,1,4614.0,1233.0,1233,1,4614.0,1233.0,1250,1,0,98.85,196.34,4.92483,9.38149,NaN,NaN,0,26003,464.339286,71,1.267857,26352,470.571429,62,1.107143,117,0.178388,0.221269,5790.550812,1,0.527158,583,820
1,p001,findYummy,417,208.0,2,3779.0,100.0,417,2,3779.0,208.0,483,2,0,95.65,180.73,4.05636,9.28450,NaN,NaN,0,5834,364.625000,20,1.250000,6002,375.125000,12,0.750000,120,0.041667,0.291667,6731.056435,2,0.082667,600,600
2,p001,frog,748,249.0,3,7595.0,267.0,748,3,7595.0,249.0,831,3,0,181.28,192.84,4.19143,9.92023,NaN,NaN,0,54661,587.752688,170,1.827957,55480,596.559140,137,1.473118,161,0.018322,0.251221,34923.741032,3,-0.128971,2456,2484
3,p001,frogInBathroom,350,350.0,1,12726.0,350.0,350,1,12726.0,350.0,383,1,0,71.88,100.34,4.10084,9.31388,NaN,NaN,0,40811,559.054795,126,1.726027,41513,568.671233,99,1.356164,132,0.016667,0.326190,15058.935995,1,-0.237870,1260,1260
4,p001,headphoneInBathroom,2572,643.0,4,3418.0,200.0,2605,2,3418.0,1303.0,2672,2,2,36.89,91.30,4.20997,9.63355,NaN,NaN,0,29902,474.634921,87,1.380952,30318,481.238095,74,1.174603,122,0.311667,0.285000,7344.425598,4,0.010817,600,634


## 5. Feature summary

In [14]:
print("Feature columns:")
feature_cols = [c for c in features.columns if c not in ['participant_id', 'task']]
print(f"  Total: {len(feature_cols)}")
for c in feature_cols:
    print(f"  {c}")

Feature columns:
  Total: 35
  correct_Total_duration_of_fixations
  correct_Average_duration_of_fixations
  correct_Number_of_fixations
  correct_Time_to_first_fixation
  correct_Duration_of_first_fixation
  correct_Total_duration_of_Visit
  correct_Number_of_Visits
  correct_Time_to_first_Visit
  correct_Average_duration_of_Visit
  correct_Total_duration_of_Glances
  correct_Number_of_Glances
  correct_Number_of_saccades_in_AOI
  correct_Peak_velocity_of_entry_saccade
  correct_Peak_velocity_of_exit_saccade
  correct_Average_pupil_diameter
  correct_Average_eye_openness
  correct_Time_to_first_mouse_click
  correct_Time_from_first_fixation_to_mouse_click
  correct_Number_of_mouse_clicks
  distractor_sum_Total_duration_of_fixations
  distractor_mean_Total_duration_of_fixations
  distractor_sum_Number_of_fixations
  distractor_mean_Number_of_fixations
  distractor_sum_Total_duration_of_Visit
  distractor_mean_Total_duration_of_Visit
  distractor_sum_Number_of_Visits
  distractor_mean_N

In [15]:
null_summary = features[feature_cols].isnull().mean().sort_values(ascending=False)
print("\nNull rate per feature:")
null_summary[null_summary > 0]


Null rate per feature:


correct_Time_to_first_mouse_click                  1.000000
correct_Time_from_first_fixation_to_mouse_click    1.000000
correct_Peak_velocity_of_exit_saccade              0.410714
correct_Peak_velocity_of_entry_saccade             0.357143
correct_Average_eye_openness                       0.321429
correct_Average_duration_of_fixations              0.321429
correct_Average_pupil_diameter                     0.321429
correct_Time_to_first_fixation                     0.321429
correct_Duration_of_first_fixation                 0.321429
correct_Average_duration_of_Visit                  0.321429
correct_Time_to_first_Visit                        0.321429
dtype: float64

## 6. Save

In [16]:
features.to_parquet(FEATURES_PER_TASK, index=False)
print(f"Saved: {FEATURES_PER_TASK}")
print(f"Shape: {features.shape}")

Saved: /home/wlsoo/WID2003/data/processed/features_per_task.parquet
Shape: (56, 37)
